# 用原生 ARTIQ 准备原子，用 Spectrum 做 578 nm 钟谱

**一个频点的一次 shot**：装载 Green MOT → 灰色光学黏胶冷却（GM）→
态制备（本课选 556 nm 光抽运）→ 578 nm 探测脉冲 → 399 nm 成像读出。
ARTIQ core 执行原生 kernel、安排触发，Spectrum AWG 产生探测链路的电波形。
上位机在 shot 外准备波形、接收图像，频率扫描的每次重复都重新装载原子。

**第一次阅读**：在 TaiyiDSL 源码目录运行
`uv sync --locked --group notebooks`。用 VS Code 打开本文件，选择项目
`.venv` 的 Python 3.12 内核，再选择「重启内核 → 全部运行」。
默认单元使用随仓库附带的 ARTIQ 教学类，可直接看到导出的 kernel、
脉冲表和各频点的最终 DAC 样本，不需要登录 GitHub 或接入仪器。

**接入自己的工程**：先读下方的源码对照表。在配置单元填入本地
`atom-control/artiq-master` 路径，再执行设备库和生产适配章节。
输入文件保留在原 MOT 工程，导出和验证材料集中在
`output/native_lab_tutorial/`。建议第一次在离线 checkout 上练习，
原实验仍可在自己的 VS Code 窗口调试。

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/taiyidsl").is_dir() and (p / "examples").is_dir()
)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUT = ROOT / "output/native_lab_tutorial"
OUT.mkdir(parents=True, exist_ok=True)

# 可直接改为 Path(r"D:\lab\atom-control\artiq-master")。
# 也可在启动 Jupyter 之前设置环境变量 TAIYI_ATOM_MASTER。
atom_path = os.environ.get("TAIYI_ATOM_MASTER")
ATOM_MASTER = Path(atom_path).expanduser().resolve(strict=True) if atom_path else None
if ATOM_MASTER is not None and not (ATOM_MASTER / "device_db.py").is_file():
    raise ValueError(
        "请选择同时包含 device_db.py、lib 和 repository 的 artiq-master 目录"
    )
print("Python", sys.version.split()[0], "| 教程输出", OUT)
print("生产源码", ATOM_MASTER or "未指定，本次运行教学实例")

## 先确认要复用的是哪一段

下表按 `production` 的固定提交 `e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e` 阅读。实验台若有本地修改，
先对照实际文件，再选阶段。特别留意两份不同的文件都定义了 `StatePrep`。

| 参考文件 | 它在这一版实际做什么 | 放入组合 shot 的部分 |
|---|---|---|
| [GreenMOT.py](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/GreenMOT.py) 和 [基类](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/lib/_GreenMOTbase.py) | 入口负责初始化、扫描、可选成像和收尾；装载、压缩与关场在基类 | 复用 `prepare_GreenMOT()` 及调用链，单独安排初始化和逐 shot 收尾 |
| [GrayMolasses.py](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/GrayMolasses.py#L52-L76) | `StatePrep.run()` 在 MOT 后设置 556 DDS、偏置场、开关光并恢复偏置，随后飞行成像 | 从内层循环提取 GM 段，保留两端的场稳定等待；MOT 和成像各执行一次 |
| [Stateprepartion.py](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/Stateprepartion.py#L138-L194) | `_sequence()` 扫描钟 AOM；`_shot()` 明确跳过 556 光抽运、RF 门和 blow-away | 用来对照 OFF/ON、偏置场、钟光门和 399 三帧读出 |
| [Stateprepartion_Spectrum.py](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/Stateprepartion_Spectrum.py#L208-L231) | 先用 556 光抽运，再触发 Spectrum 的 RF 门，随后做 blow-away | 本课只选光抽运作为 state prep。若实验定义还要求 RF 转移、清除未制备原子，请把这两步及其资源、时序另行纳入 |

实验台原 578 光路使用 `AOM_578_Clock`（卫星 Urukul 的 `urukul6_ch3`）。
`AOM_578_Shift` 是另一设备。上述 Spectrum 脚本则用卡输出 RF 自旋门。
本课选择让 **Spectrum 的模拟输出经实验室已标定的 RF/IF 链路驱动 578 探测**。
如果原接线仍由 Urukul 独占钟 AOM，先保留教学模式；仅改信号名称不能改变物理路由。

## 把实验台的设备名称接到混合后端

**先画出两根线。** 一根是 Kasli 的某路空闲 TTL → Spectrum 的外触发输入。
另一根是 Spectrum 的某路 DAC → 实验室实际用于 578 nm 探测的射频链。
确认卡号、输出端口、触发极性、参考时钟、射频开关和光学通路，再把它们写入接线描述。
ARTIQ 的 `device_db` 定义 core、TTL、DDS、Zotino 和别名。
Spectrum 卡的身份与设置放在 `spec_db`。
`descriptor` 决定逻辑信号 `q.clock` 最终送到哪个物理端口。

所引生产版本把 `urukul6_ch3` 指向 `AOM_578_Clock`，把 `urukul0_ch3` 指向
`AOM_578_Shift`。前者是原生钟谱程序使用的卫星 DDS。
混合示例选择 Spectrum 作为钟脉冲波形源，需要核对同轴接线。
原来的钟 DDS 如保留作本振或固定移频，
要单独约定它的工作点、开关及归属，避免两条控制路径同时驱动同一输入。数据库中的
`ttl6` 是 `ttl_Camera_Trig` 的别名。`ttl_RF_Gate_Trig` 已在原程序中使用。
两者都不能仅凭名字选作空闲的 AWG 触发线。
[原始设备库](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/device_db.py)

下面的函数在本机解析文件。先用教学卡 B、DAC 0 和教学 TTL 演练。
查看 `connections` 和 `trigger_connections`，
分别按“信号 → 端口”和“TTL → 卡”对照接线。

In [ ]:
from taiyidsl import DeviceSetup


def build_clock_setup(device_db, spec_db, *, card_id, clock_port, trigger_name):
    descriptor = {
        "name": "clock_spectroscopy",
        "backend": "artiq-spectrum",
        "instruments": {
            "artiq": {"kind": "artiq", "device": "core"},
            card_id: {"kind": "spectrum", "device": card_id},
        },
        "channels": {
            "clock_drive": {"instrument": card_id, "channel": clock_port},
        },
        "signals": {"q": {"clock": "clock_drive"}},
        "sync_groups": {
            "clock_awg": {"cards": [card_id], "trigger": trigger_name},
        },
    }
    return DeviceSetup.from_descriptor(descriptor, device_db=device_db, spec_db=spec_db)


example_dbs = ROOT / "examples" / "artiq_spectrum" / "ramsey_lab"
wiring_example = build_clock_setup(
    example_dbs / "device_db.py",
    example_dbs / "spec_db.py",
    card_id="B",
    clock_port=0,
    trigger_name="awg_trigger_b",
)
print("逻辑信号和物理端口", wiring_example.connections())
print("触发输出和目标卡", wiring_example.trigger_connections())

### 将生产 `device_db.py` 变为可静态检查的快照

生产文件包含 `import os`、`ORCA_*` 环境变量、逐项写入和卫星编号等动态设置。
TaiyiDSL 的文件读取器只处理声明式字典。**在运行原项目的可信配置环境中**，
核对 checkout、ORCA 环境与 master 一致后，用 PowerShell 读取设备定义。
该命令会执行你指定的 Python 配置文件，应先审查本地版本。
用实际路径替换示例路径。输出到实验室私有目录，
保留整数卫星编号和别名，并在首行记录原始文件哈希。

```powershell
$db = 'C:/lab/atom-control/artiq-master/device_db.py'
$snapshot = 'C:/lab/private_config/device_db_snapshot.py'
@'
import hashlib
import pprint
import runpy
import sys
from pathlib import Path

source, output = map(Path, sys.argv[1:])
digest = hashlib.sha256(source.read_bytes()).hexdigest()
devices = runpy.run_path(str(source))["device_db"]
body = pprint.pformat(devices, sort_dicts=False)
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    f"# source_sha256={digest}\ndevice_db = {body}\n", encoding="utf-8"
)
'@ | python - $db $snapshot
```

在同一私有目录准备 `spec_db.py`，按教学文件填写实测卡的序列号、型号、
参考时钟、采样率、启用通道、输出量程和触发延迟；另建 `clock_route.toml`，
填写 `card_id`（与 `spec_db` 的键一致）、`clock_port`（整数）和
`trigger_name`（经现场确认空闲、已接外触发的设备名）。
本例的混合后端支持 M4i.6631-x8、1.25 GS/s 与 125 MHz 参考。
触发线以外，三段原生代码使用的 DDS、共享 CPLD/SPI、DAC、相机等也必须由同一份
设备库解析。它们由原生模块声明。
专用于原生 kernel 的设备通常不用逐个加到 DSL 的 `channels`。
配置快照含实验室网络与仪器身份，只在实验室保存；环境变量改变时要重新生成。
[设备描述与同步校准指南](../device_setup_descriptor.md)

In [ ]:
import tomllib
from hashlib import sha256
from pathlib import Path

# 可选。填好 ATOM_MASTER 后，再将下一行换成本机保存三份文件的目录。
PRIVATE_CONFIG = None  # 例如 Path("C:/lab/private_config")
reviewed_clock_setup = None
if PRIVATE_CONFIG is not None:
    if ATOM_MASTER is None:
        raise ValueError("ATOM_MASTER 应指向含 device_db.py 的目录")
    private_dir = Path(PRIVATE_CONFIG)
    original_db = ATOM_MASTER / "device_db.py"
    snapshot = private_dir / "device_db_snapshot.py"
    original_hash = sha256(original_db.read_bytes()).hexdigest()
    with snapshot.open(encoding="utf-8") as stream:
        if stream.readline().strip() != "# source_sha256=" + original_hash:
            raise ValueError("设备库来源不符，请重新核对并导出快照")
    with (private_dir / "clock_route.toml").open("rb") as stream:
        route = tomllib.load(stream)
    reviewed_clock_setup = build_clock_setup(
        snapshot, private_dir / "spec_db.py", **route
    )
    print("设备库快照 SHA-256", sha256(snapshot.read_bytes()).hexdigest())
    print("核对现场接线", reviewed_clock_setup.connections())
    print("核对现场触发", reviewed_clock_setup.trigger_connections())

### `experiment_config` 给谁读？从调用者追到字段

在所引版本，三个 20261120 实验及其父类 `lib/_GreenMOTbase.py` 通过
`setattr_argument`、`prepare()` 和代码常量组织参数；
这些程序没有直接加载六个 TOML。
例如，`GrayMolasses.py` 的 `gray_molasses_preparation_freq` 来自 ARTIQ 的
`Scannable`，继承的 `T_compress` 来自父类 `build()`。保存本次提交所选的
具体值，确认扫频单位是 Hz、持续时间是秒后，再交给适配层。
`repository/helpers/mot_3d_config.py` 的 `prepare_mot(exp, config_dir)`
会读取 `laser_alignment_config.toml`、`coil_config.toml`、
`MOT_3D_config.toml`，定位 AOM 并计算该版 3D MOT 的参数。
这是另一条可以独立复用的准备路径。
[配置目录](https://github.com/TaiyiQ/atom-control/tree/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/experiment_config)
[读取三个配置的函数](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/helpers/mot_3d_config.py)

| 文件 | 在这个版本中的内容与用途 |
| --- | --- |
| `MOT_3D_config.toml` | 399 装载、交接和 556 捕获、冷却、压缩。 |
| `coil_config.toml` | 线圈设定电压；换算电流须用实际驱动器系数。 |
| `laser_alignment_config.toml` | 各 Urukul 射频工作点与通道用途。 |
| `laser_scan_config.toml` | 激光扫描规则，固定工作点取自 alignment 配置。 |
| `bias_magnetic_field_scan_config.toml` | Zotino 三轴偏磁扫描。 |
| `absorption_imaging_config.toml` | 此版本为空，没有相机参数。 |

下格只列出本地配置的顶层键、字节数和哈希，用来检查来源及版本差异。确定需要复用某个
配置后，在 host 端读取具体键并按文件的单位转换。示例将另一条 3D MOT 路径的
`coil_ramp_ms` 转成秒并验证数值；这些值不会自动注入下文的 GM 或 state prep。

In [ ]:
if ATOM_MASTER is not None:
    config_dir = ATOM_MASTER / "experiment_config"
    for config_path in sorted(config_dir.glob("*.toml")):
        raw = config_path.read_bytes()
        sections = tomllib.loads(raw.decode("utf-8-sig"))
        digest = sha256(raw).hexdigest()
        print(config_path.name, len(raw), digest, list(sections))
    with (config_dir / "MOT_3D_config.toml").open("rb") as stream:
        mot_config = tomllib.load(stream)
    coil_ramp_s = float(mot_config["sequence"]["coil_ramp_ms"]) * 1e-3
    assert coil_ramp_s > 0
    print("已验证 coil_ramp_ms 的单位转换（仅检查另一条 3D MOT 准备路径）")

## 在离线实验台上走完一个 shot

先用仓库自带的四个 ARTIQ 类练习接线、注册与定时。三个准备阶段共用一根 TTL，
第四个类给相机发触发脉冲。它们的实现很短，适合与生成的顶层 kernel 对照阅读。
这里把 Spectrum 卡 B 的 CH0 接到假想的 578 nm 探测链路，
由另一根 ARTIQ TTL 触发卡 B 的 EXT0。
上游用 `AOM_578_Clock` 指向 `urukul6_ch3`。实验室若改用 Spectrum 驱动
这条光路，须确认实际 RF 接线和切换方式，查明旧 DDS 驱动何时退出。
运行这些单元只会读取源码、编译并写本地文件。

| 一条信号在哪里定义 | 此例中可以核对的内容 |
|---|---|
| ARTIQ `device_db` | core、原生阶段的两个 TTL、AWG 的触发 TTL |
| Spectrum `spec_db` | 卡标识、采样率、参考时钟、输出范围及示例触发延迟 |
| `descriptor` | `q.clock → clock_drive → B/CH0`、`awg_trigger_b → B/EXT0` |
| `Qubit.parameters.clock` | 教学探测脉冲的电载波中心频率 |

本例沿用教学设备库中的卡 B，只启用一个输出。合并两份**教学**设备表前检查重名；
上机时从实验室的完整设备表读取，逐根核对 TTL、别名与物理接线。

In [ ]:
import json
from pprint import pprint

import numpy as np
from IPython.display import Image, display

from examples.artiq_spectrum.ramsey_lab.device_db import device_db as example_db
from examples.artiq_spectrum.ramsey_lab.spec_db import spec_db as card_db
from taiyidsl import (
    Experiment,
    LabConfig,
    NativeModule,
    NativeProgram,
    NativeStage,
    Qubit,
    Square,
    SweepParameter,
    delay,
    kHz,
    play,
    us,
)

native_root = ROOT / "examples" / "artiq_spectrum"
native_dir = native_root / "native_stages"
# 此表只补充教学原生类要用的两根 TTL。覆盖重名项会丢失原有接线。
extra_db_file = native_dir / "device_db_offline.json"
native_db = json.loads(extra_db_file.read_text(encoding="utf-8"))
assert native_db.keys().isdisjoint(example_db)
fixture_db = example_db | native_db
fixture_cards = {"B": {**card_db["B"], "active_channels": [0]}}

descriptor = {
    "name": "offline_clock_spectroscopy",  # 接线方案名，便于追溯。
    "backend": "artiq-spectrum",  # core 排时，Spectrum 生成 DAC 样本。
    "instruments": {
        # 左侧是 descriptor 内的名字，device 是相应设备库中的键。
        "rtio": {"kind": "artiq", "device": "core"},
        "B": {"kind": "spectrum", "device": "B"},
    },
    "channels": {
        # channel=0 是卡 B 的模拟 CH0。触发通道则引用 ARTIQ 设备名。
        "clock_drive": {"instrument": "B", "channel": 0},
        "clock_trigger": {"instrument": "rtio", "device": "awg_trigger_b"},
    },
    "sync_groups": {
        # 后文用 group="clock_awg" 调用本组。B 的 EXT0 接 clock_trigger。
        "clock_awg": {"cards": ["B"], "triggers": {"B": "clock_trigger"}},
    },
    "signals": {"q": {"clock": "clock_drive"}},  # 逻辑角色 → 物理输出。
}
setup = DeviceSetup.from_descriptor(
    # descriptor 定义映射，device_db 解析 core/TTL，spec_db 解析卡 B。
    descriptor,
    device_db=fixture_db,
    spec_db=fixture_cards,
)
# "q" 是实例 ID；signals 绑定角色；frequency 是假定的电载波中心。
q = Qubit(
    "q", signals={"clock": "q.clock"}, parameters={"clock": {"frequency": 200 * kHz}}
)
# setup 提供接线，qubits 注册实例，emulation=True 启用离线模式。
lab = LabConfig(setup=setup, qubits=[q], emulation=True)
pprint(setup.connections())
pprint(setup.trigger_connections())

### 读一个类，再注册四个组件

下面直接显示本次要注册的四个源文件。先对照 `MOT.build()` 里绑定的设备名，
再看 `prepare_MOT()` 里的 `break_realtime()` 和 GM 方法的最后一个边沿。
原工程接入时也从这两处入手。类里声明的每个外设都应能在实验室设备库里解析。

In [ ]:
for filename in ("mot.py", "cooling.py", "state_prep.py", "readout.py"):
    print(f"—— {filename} ——")
    print((native_dir / filename).read_text(encoding="utf-8"))

`root` 指向**包含**顶层 Python 包 `native_stages` 的目录；`entry` 写成
`包.模块:类`。这一步读取源码并固定哈希。实验定义里的 `native.GM_cooling(...)`
只会记录未来要执行的方法和参数。导出的顶层 kernel 会在对应位置调用它。

一个模块对应一个 ARTIQ 实例。这里让 MOT 独占共享准备线的初始化；
cooling 和 state prep 可以按阶段操作同一根线。相机有独立 TTL。
`initialize` 在原生会话开始时执行，`after_shot` 在每个完整 shot 后执行，
`finalize` 在会话结束时执行。真实组件还需梳理共享 DDS、CPLD 和总线的初始化归属。

In [ ]:
# ① 每个声明描述一个类。from_package 在注册时读取源码与依赖。
mot = NativeModule.from_package(
    root=native_root,  # 包的父目录，本例是 examples/artiq_spectrum。
    entry="native_stages.mot:MOT",  # native_stages/mot.py 中的 MOT 类。
    resources=("native_prepare",),  # MOT 使用的外设名；core 隐式共享。
    # MOT.build(self) 没有配置形参，故 build_kwargs 留空。
    # 有 build(self, mode) 时可传 build_kwargs={"mode": "green"}。
    # 它固定于本次实例构建，不能在这里传 SweepParameter。
    initialize="initialize",  # 首个 shot 前调用一次，准备线置低。
    after_shot="after_shot",  # 每个完整 shot 后使准备线恢复低电平。
    finalize="finalize",  # 会话结束时调用一次，恢复准备线。
    # 若要读静态宿主配置，可设 host_prepare="实际宿主方法名"。
    # 该钩子每个会话一次。本例没有宿主准备方法，故省略。
    # 此包的静态依赖都在 root 内。若有根目录外的受审源码，可用
    # source_overrides={"模块名": 源文件路径} 补充，再重新导出。
)
# ② Cooling 实例复用准备线。省略 initialize，使 MOT 负责初始化。
cooling = NativeModule.from_package(
    root=native_root,  # 与 MOT 属于同一源码包。
    entry="native_stages.cooling:Cooling",  # 模块 cooling 中的类。
    resources=("native_prepare",),  # GM 方法会拉高再拉低这根线。
)
# ③ 第三个实例也使用准备线；它有自己的方法和实例变量。
state = NativeModule.from_package(
    root=native_root,
    entry="native_stages.state_prep:StatePreparation",  # 态制备类。
    resources=("native_prepare",),
)
# ④ Readout 单独初始化相机 TTL。它发触发，并读取控制末端参考系相位。
readout = NativeModule.from_package(
    root=native_root,
    entry="native_stages.readout:Readout",  # readout.py 中的读出类。
    resources=("native_camera",),  # 与准备 TTL、AWG 触发都不同。
    initialize="initialize",  # 相机 TTL 在首个 shot 前置低。
)

### 给四个可调用阶段标上时序边界

MOT 含有 `break_realtime()`，把它作为 `prefix`。组合器在最后一个 prefix
返回后恢复调度余量并建立时间原点。教学 GM 与制备各执行一次定长 TTL 脉冲，
因此可分别声明为 20 µs、8 µs 的 `timed`。读出在控制段结束后执行，声明为
10 µs 的 `suffix`。这些长度也作为方法实参传入，顶层 kernel 校验返回游标。
将生产方法映射到 `timed` 前，要审查其完整调用链，含等待、RPC 或
`break_realtime()` 的方法应放在 prefix 或重新划分边界。

In [ ]:
gm_hold, state_hold, camera_width = 20 * us, 8 * us, 10 * us
# modules 建立“组件名 → 声明”的表，供 NativeStage.component 引用。
# 程序级 initialize/after_shot/finalize/host_prepare 均可指定组件名
# 元组，用来选择并排序已声明的钩子。这里省略，按 modules 的顺序执行。
native = NativeProgram(
    modules={"mot": mot, "gm": cooling, "state": state, "camera": readout},
    # stages 的键是 DSL 中 native.<键>(...) 使用的名字。
    # 各阶段的实际调用先后由下一个代码单元的 realtime 主体确定。
    stages={
        "prepare_MOT": NativeStage(
            component="mot",  # 从 modules 选 MOT 实例。
            method="prepare_MOT",  # 调用该类上的同名 @kernel。
            timing="prefix",  # 可变耗时；省略 duration 和 end_boundary。
        ),
        "GM_cooling": NativeStage(
            component="gm",  # 查找上面的 cooling 声明。
            method="GM_cooling",  # 真正被调用的方法，接收 duration。
            timing="timed",  # 在 DSL 控制段之前精确定时。
            duration=gm_hold,  # 校验从调用到返回共 20 µs。
            # end_boundary 说明把哪个事件认作结束，由工程师核实。
            end_boundary="native_prepare falling edge after GM",
        ),
        "state_prep": NativeStage(
            component="state",  # 与 modules 中的组件名一致。
            method="state_prep",  # 类方法接收 duration。
            timing="timed",  # 从前一个 timed 阶段的末端接续。
            duration=state_hold,  # 声明游标前进 8 µs。
            # 这里的边界是数字输出，不包含额外的模拟场稳定时间。
            end_boundary="native_prepare falling edge after state preparation",
        ),
        "readout": NativeStage(
            component="camera",  # 选择独立的 Readout 实例。
            method="execute_readout",  # DSL 名为 readout，类方法可不同。
            timing="suffix",  # 在 DSL 控制段结束后调用。
            duration=camera_width,  # 仅指 10 µs 相机 TTL 脉宽。
            # 此方法读取 native_terminal_f_q，即控制末端的参考系相位。
            # 前三阶段使用默认 False；普通触发可不依赖相位。
            phase_sensitive=True,
            end_boundary="native_camera falling edge",  # TTL 下降沿。
        ),
    },
    # 键是设备名，值列全使用它的组件。只给 mot 注册了准备线初始化。
    # 漏列 gm、或给 gm 也加 initialize，都会导致共享资源校验失败。
    shared_resources={"native_prepare": ("mot", "gm", "state")},
)

### 写 578 nm 单脉冲频谱扫描

每个失谐点都重新装载、冷却、制备、探测并触发读出。示例取五个**光学失谐**
Δν，假设从 AWG 电频率到光学频移的有符号斜率 α 为 +1 Hz/Hz，示例中心为
200 kHz 电载波，故命令为 `f_AWG = f_center + Δν / α`。
578 nm 表示目标跃迁的光波长。实机要用已标定的 RF/IF 中心和变频关系，
按所选衍射级、通光次数和混频边带确认符号与倍率；下列 200 kHz 和 +1
仅用于演示换算。频率扫描只施加一次。

原生调用直接写在 `realtime()` 内；仅探测脉冲写在 `triggered_sequence`
内。state prep 的末边沿与探测脉冲之间预留 10 µs，供本例触发提前量使用。
真实电路还需核实模拟场稳定时间和光路传播延迟。固定探测宽度和幅度，便于
把观测量按光学失谐归档。每点做两次 shot，用于检查索引。

In [ ]:
def clock_spectroscopy(
    q, native, probe, awg_frequency, *, gm_time, state_time, guard, readout_time, shots
):
    # q 是已注册的 qubit，native 是上一步具名阶段的集合。
    exp = Experiment("native_578_clock_spectroscopy", [q])
    # average 规定每点 shot 数；sweep 绑定本点频率；realtime 包住一发。
    with exp.average(shots), exp.sweep(awg_frequency), exp.realtime():
        # 类方法 prepare_MOT(self) 只有 self，调用者传零个实参。
        native.prepare_MOT()
        # 按方法签名绑定到 GM_cooling(self, duration) 的 duration。
        # 这是每次调用的实参；NativeStage.duration 只负责核对游标。
        native.GM_cooling(gm_time)
        # state_prep 同样以第一个位置实参接收教学 TTL 的高电平时长。
        native.state_prep(state_time)
        # 让上一个 timed 阶段结束后有明确空档，留给 AWG 提前触发。
        delay(guard)
        # group 对应 descriptor.sync_groups 的键，触发路由由它决定。
        with exp.triggered_sequence(group="clock_awg"):
            # 信号选卡口，probe 给包络，frequency 覆盖本点电载波。
            play(q.clock, probe, frequency=awg_frequency)
        # 回到 realtime 的直接子层，绑定 execute_readout(duration)。
        # 只发相机 TTL。曝光、取帧与状态判别属于实验室读出链。
        native.readout(readout_time)
    return exp


optical_detuning_hz = [-20_000, -10_000, 0, 10_000, 20_000]
optical_hz_per_awg_hz = +1.0
center_hz = q.parameters.clock.frequency
drive_hz = [center_hz + delta / optical_hz_per_awg_hz for delta in optical_detuning_hz]
# 扫描参数的名字、每点电频率列表、列表数值的单位。
frequency_axis = SweepParameter("awg_frequency", drive_hz, "Hz")
# 方形包络的持续时间与归一化幅度；每点的频率在 play 时给出。
probe = Square(duration=40 * us, amplitude=0.2)
exp = clock_spectroscopy(
    q,
    native,
    probe,
    frequency_axis,
    # 三个原生方法的时长实参复用前文合同值，guard 属于控制空档。
    gm_time=gm_hold,
    state_time=state_hold,
    guard=10 * us,
    readout_time=camera_width,
    shots=2,
)
# 修改 gm_hold 后，须重建 native 和 exp 再 compile，否则旧合同仍在。
# 单改调用实参会使实际游标与旧合同不符，运行至边界校验时会报错。
compiled = lab.compile(exp)
execution_order = list(compiled.execution_order())
print(
    "扫描点、每点 shot 数、总 shot 数：",
    len(compiled.points),
    compiled.shots,
    len(execution_order),
)

### 核对 AWG 输出与顶层 kernel

从每个编译点取出探测事件的实际频率，再从最终的分段表回放探测前 8 µs 的整数
DAC 样本。下图时间以探测脉冲的逻辑起点为零，纵轴是相对于满量程的输出。
运行后应看到五个不同频率，且各点的实际样本不同。此图可发现
“只扫了表格标签、没有改变硬件指令”这类接线或参数错误。原子响应需要现场采集。

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

assert len(compiled.points) == len(drive_hz)
assert len(execution_order) == len(drive_hz) * compiled.shots
fig, ax = plt.subplots(figsize=(8, 3.4))
samples = []
for point, expected, delta in zip(
    compiled.points, drive_hz, optical_detuning_hz, strict=True
):
    (event,) = [e for e in point.logical_events if e.kind == "play"]
    assert event.signal == q.clock and event.params["frequency"] == expected
    start_s = event.start_mu * float(compiled.setup.tick_seconds)
    window = compiled.simulate_output(
        point=point.index, start_s=start_s, duration_s=8e-6
    )
    (trace,) = [t for t in window["traces"] if (t["card"], t["channel"]) == ("B", 0)]
    assert np.any(trace["dac_codes"])
    samples.append(trace["dac_codes"].tobytes())
    ax.plot(
        (trace["time_s"] - start_s) * 1e6,
        trace["normalized"],
        label=f"{delta / 1000:+g} kHz",
    )
    print(f"point {point.index}: Δν {delta:+g} Hz, AWG {expected:g} Hz")
assert len(set(samples)) == len(drive_hz)
ax.set(xlabel="Time from probe start (us)", ylabel="DAC / full scale")
ax.legend(title="Optical offset", loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
figure_path = OUT / "clock_probe_waveforms.png"
fig.savefig(figure_path, dpi=140)
plt.close(fig)
display(Image(filename=str(figure_path)))

In [ ]:
export_dir = OUT / "export"
compiled.export(export_dir)
compiled.visualize(OUT / "pulse_sheet.html", waveform_path="waveform.html")
compiled.waveform_simulator(OUT / "waveform.html")
hybrid = json.loads((export_dir / "manifest.json").read_text(encoding="utf-8"))
core_manifest = export_dir / "artiq/manifest.json"
artiq_manifest = json.loads(core_manifest.read_text(encoding="utf-8"))
composition = artiq_manifest["native_composition"]
assert hybrid["backend"] == "artiq_spectrum"
assert hybrid["verification"]["offline"] == "passed"
assert composition["execution"] == "one_top_level_kernel_per_shot"
first = composition["points"][0]
assert [s["stage"] for s in first["prefix"]] == ["prepare_MOT"]
assert [s["stage"] for s in first["timed"]] == ["GM_cooling", "state_prep"]
assert [s["stage"] for s in first["suffix"]] == ["readout"]
for section in ("prefix", "timed", "suffix"):
    pprint({section: first[section]})
print("生成的 ARTIQ 顶层 kernel：", export_dir / "artiq/experiment.py")
print("脉冲表：", OUT / "pulse_sheet.html")
print("最终 DAC 波形：", OUT / "waveform.html")

在 `export/artiq/experiment.py` 中搜索 `def point_0(`，
按顺序找到 MOT、恢复调度余量、GM、state prep、控制段调用和相机触发。
`prefix` 的耗时没有固定值，所以不能从 pulse sheet 倒推出 MOT 开始时刻。
查看 `export/experiment.py` 可找到混合实验的宿主入口。交付时应保留完整
`export/`、匹配的原生 Python 源和实验室的读出适配器；在实际 worker 里让
`native_stages` 或替换后的实验室包可导入。

真机采集要记录 point、shot、电载波、光学失谐和帧 ID。
根据已验证的态选择读出方式计算每点激发概率，再给出有效 shot 数与不确定度。
本例中的 `native_camera` 只发 TTL；此处的十个计划 shot 没有任何实测激发概率。

## 用实验室原代码组成同一个 shot

这一节是可选的真实源码练习。在开头将 `ATOM_MASTER` 指向自己的
`atom-control/artiq-master` 后再运行。本节只读源文件，把适配层写入本教程的
`output/`，不会导入 ARTIQ、执行原实验的 `run()` 或连接仪器。
这些代码可以在自己的实验室包中维护。复制前先打开下表的源码段，核对边界。

| 本教程阶段 | 这份固定版本中实际执行的代码 | 需要从原实验入口移出的部分 |
|---|---|---|
| MOT | `GreenMOT.run` 的一次装载调用基类 `prepare_GreenMOT()` | 扫描、成像和循环后的复位由新的 shot 统一组织 |
| GM | `GrayMolasses.py` 的类名是 `StatePrep`，`run()` 第 60–68 行才是 GM 脉段：等偏置场、556 光照 4 ms、关光、恢复 EW 场后再等待 | 外围的第二次装载、TOF 和成像 |
| 556 态制备 | 同目录 `Stateprepartion_Spectrum.py` 第 209–217 行包含实际运行的光抽运：施加偏置、等待、设置探测 AOM、开光、关光、等待消光 | 后面的 RF 门与 blow-away 有独立物理作用；本例只取光抽运，须单独评估目标态布居 |
| 578 nm 探测 | 在混合后端用 DSL 产生探测波形 | `Stateprepartion.py` 本身是原生 578 DDS 钟谱；其 `_shot` 明确跳过光抽运、RF 门和 blow-away |
| 399 nm 读出 | 取原钟谱 `_shot` 第 187–193 行：TOF、恢复偏置、等待、设置 399 探测 DDS、水平三帧和收尾 | 一次一组，实际图像、帧 ID 与数据提交仍由采集端负责 |

下列代码从经核对的文件提取原句，以便保留 DDS、Zotino 与快门的指令顺序。指令起止行严格对应提交
`e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e`。
对于新的生产版本，请重新逐段比对，确认物理顺序后再更新源指纹、起止行及验证记录。
[MOT](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/GreenMOT.py)
· [GM](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/GrayMolasses.py#L60-L68)
· [光抽运](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/Stateprepartion_Spectrum.py#L209-L217)
· [原生钟谱与读出](https://github.com/TaiyiQ/atom-control/blob/e2eedceeae6408bfb1c8f6aa7c4da3dfc4184d2e/artiq-master/repository/20261120/Stateprepartion.py#L170-L193)

In [ ]:
import ast
import hashlib
from pathlib import Path
from textwrap import dedent, indent

date_folder = Path("repository/20261120")
base_file = Path("lib/_GreenMOTbase.py")
gm_file = date_folder / "GrayMolasses.py"
pump_file = date_folder / "Stateprepartion_Spectrum.py"
clock_file = date_folder / "Stateprepartion.py"
reviewed_blobs = {
    base_file: "91f2d9e606b8818c43aca8e4b219abf1c2bf6c68",
    Path("lib/__init__.py"): "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391",
    date_folder / "GreenMOT.py": "74059af6ba809801945165ec43fa265a40302ab3",
    gm_file: "1638d23b02eb5362dc43c11b8b3667a7a3579365",
    pump_file: "4bc3b56983ef084a83e7c07932e993501218b5d3",
    clock_file: "eb5702e62d4a5ffd7f782a651ec92eba8a322cc3",
}


def reviewed_sources(master):
    sources = {}
    for path, expected in reviewed_blobs.items():
        source = (master / path).read_text(encoding="utf-8")
        data = source.encode("utf-8")  # Git blob 采用 LF；支持 Windows checkout
        blob = hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data)
        if blob.hexdigest() != expected:
            raise ValueError(f"{path} 已变化，请重新审核其调用和对应行。")
        sources[path] = source
    return sources


def region(source, first, last):
    return dedent("\n".join(source.splitlines()[first - 1 : last])).strip()


if ATOM_MASTER is not None:
    atom_sources = reviewed_sources(Path(ATOM_MASTER))
    print(region(atom_sources[pump_file], 209, 217))
else:
    print("如要生成真实源码适配层，请先设置 ATOM_MASTER。")

**一处初始化，共享偏置场。** MOT 继承原基类；只将 `init_hardware()` 开头的
`core.reset()` 移交给生成实验的根初始化，组件用 `break_realtime()` 为随后的 CPLD 首次写入恢复余量，保留其余设备初始化及等待。
水平相机以下降沿触发，初始化时将 TTL 置高；556 探测开关在首次抽运前要能消光。
其他两个组件只绑定自己实际驱动的器件。原基类的 EW 基准在三者中使用同一个值。
示例从源码的参数声明读取默认值，再通过 `build_kwargs` 固定。迁移实际实验时，应使用该次运行的参数记录替换这些起点。
例如某次 Dashboard 改过的 GM 频率不会出现在源码默认值里。

三个准备段均放在 `prefix`。MOT 内部有 `break_realtime()`；GM 和光抽运除显式等待外还有驱动开销。
最后一个 prefix 后，编排器重新获得调度余量并建立控制时间原点。若要规定“实际抽运关光到 578 光到原子”的间隔，
需要拆出末段为有界的 `timed` stage，并实测设备、光路和偏置场的响应。
这里的 `guard` 在触发启动的 AWG 波形内部产生静默前导。

读出需要在 578 脉冲后执行 TOF、Zotino 和 DDS 配置。这些操作的总耗时还没有经验证的合约，
因而本节把整段安排在每个完成 shot 的 `after_shot` 中，沿用上游次序。
它在每点的顶层 kernel 内执行一次。不要再叠加教学 TTL 的 `native.readout(...)`。
若要把第一次成像沿严格锁定在某个时间，应另外设计和验证 `suffix` 的固定时长及其起始状态。

In [ ]:
def write_stage_adapter(sources, target):
    # 保留原文件，导入基类；将两个内联阶段和一个收尾段写到自己的模块中。
    # MOT.build 沿用原设备绑定，统一 EW 偏置并保存每次读出前使用的 TOF 秒数。
    # MOT.load 负责一次装载，保留原入口的调度余量和 5 ms 前导。
    header = """\
from artiq.experiment import HasEnvironment, kernel, delay, MHz, dB, ms, us
from lib._GreenMOTbase import _GreenMOTbase

class MOT(_GreenMOTbase):
    def build(self, ew_v, tof_s):
        super().build()
        self.ComField_EW_for_556MOT = ew_v
        self.tof_s = tof_s

    @kernel
    def load(self):
        self.core.break_realtime()
        delay(5 * ms)
        self.prepare_GreenMOT()

    @kernel
    def initialize_devices(self):
"""
    # 确认被替换的仅为原首句 reset，后面的板卡与 DAC 初始化逐句沿用。
    assert region(sources[base_file], 62, 62) == "self.core.reset()"
    initialize = "self.core.break_realtime()\n" + region(sources[base_file], 63, 91)
    initialize += "\nself.AOM_556_Probe.cfg_sw(False)\nself.AOM_556_Probe.sw.off()"
    initialize += "\nself.ttl_Camera_Trig.on()\ndelay(10 * us)"
    # 旧 _shot 把 tof_s 当实参；新 after_shot 没有实参，改从实例取同一个值。
    after = "tof_s = self.tof_s\n" + region(sources[clock_file], 187, 193)
    # 两个切片均只取本阶段操作，避免引入各自独立入口的循环、重复装载与成像。
    gm = region(sources[gm_file], 60, 68)
    pump = region(sources[pump_file], 209, 217)
    # GM 不再自行 new 一套 MOT；显式绑定它直接写入的器件，频率保留为阶段形参。
    gm_class = """class GrayMolasses(HasEnvironment):
    def build(self, ew_v, offset_v):
        self.setattr_device("zotino0")
        self.setattr_device("AOM_556_3DMOT")
        self.setattr_device("ttl_556_3DMOT_Sh")
        self.ComField_EW_for_556MOT = ew_v
        self.gray_EW_bias_offset = offset_v

    @kernel
    def cool(self, GM_preparation_freq):
"""
    # 光抽运通过 build 注入当次选定的频率/衰减与线圈偏置，pump 沿用旧操作。
    state_class = """class OpticalPumping(HasEnvironment):
    def build(self, ew_v, offset_v, frequency_hz, attenuation_db):
        self.setattr_device("zotino0")
        self.setattr_device("AOM_556_Probe")
        self.ComField_EW_for_556MOT = ew_v
        self.state_preparation_EW_bias_offset = offset_v
        self.f_preparation = frequency_hz
        self.att_preparation = attenuation_db

    @kernel
    def pump(self):
"""
    # 每段以 @kernel 方法写出，AST 语法检查后，用户可打开整个 .py 再审核。
    source = (
        header
        + indent(initialize, "        ")
        + "\n\n    @kernel\n    def read_and_restore(self):\n"
        + indent(after, "        ")
        + "\n\n"
        + gm_class
        + indent(gm, "        ")
        + "\n\n"
        + state_class
        + indent(pump, "        ")
        + "\n"
    )
    ast.parse(source)
    target.mkdir(parents=True, exist_ok=True)
    (target / "atom_stage_adapter.py").write_text(source, encoding="utf-8")
    return target / "atom_stage_adapter.py"

In [ ]:
def source_default(source, name):
    # 仅解析此固定版本实际使用的常量、乘法和单位；不执行文件。
    def scalar(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return float(node.value)
        if isinstance(node, ast.Name):
            return {"MHz": 1e6, "ms": 1e-3}[node.id]
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Mult):
            return scalar(node.left) * scalar(node.right)
        raise ValueError(f"参数 {name} 的默认表达式需重新审核。")

    matches = [
        n.args[1]
        for n in ast.walk(ast.parse(source))
        if isinstance(n, ast.Call)
        and isinstance(n.func, ast.Attribute)
        and n.func.attr == "setattr_argument"
        and len(n.args) >= 2
        and isinstance(n.args[0], ast.Constant)
        and n.args[0].value == name
    ]
    (processor,) = matches
    if processor.func.id == "NumberValue":
        return scalar(processor.args[0])
    assert processor.func.id == "Scannable"
    defaults = next(k.value for k in processor.keywords if k.arg == "default")
    point = defaults.elts[0]
    assert point.func.id == "NoScan"
    return scalar(point.args[0])


if ATOM_MASTER is not None:
    lab_values = {
        "ew_v": source_default(atom_sources[base_file], "ComField_EW_for_556MOT"),
        "tof_s": source_default(atom_sources[clock_file], "TOF_time"),
        "gm_frequency_hz": source_default(
            atom_sources[gm_file], "gray_molasses_preparation_freq"
        ),
        "gm_offset_v": source_default(atom_sources[gm_file], "gray_EW_bias_offset"),
        "pump_offset_v": source_default(
            atom_sources[pump_file], "state_preparation_EW_bias_offset"
        ),
        "pump_frequency_hz": source_default(atom_sources[pump_file], "f_preparation"),
        "pump_att_db": source_default(atom_sources[pump_file], "att_preparation"),
    }
    adapter_root = OUT / "production_adapter"
    print(write_stage_adapter(atom_sources, adapter_root))
    print("本次离线编排使用的源码默认值：", lab_values)

### 按自己的模块替换每项参数

| 传入的位置 | 本例的实参 | 你在自己工程中要填的内容 |
|---|---|---|
| `NativeModule.from_package(root=..., entry=...)` | `production_adapter` 和 `atom_stage_adapter:GrayMolasses` | 前者是 Python 导入根目录，后者是该根下的模块名和类名。例如文件为 `src/my_lab/gm.py` 时写 `root=src`、`entry="my_lab.gm:Cooling"` |
| `source_overrides` | `lib` 与 `lib._GreenMOTbase` 的工作副本文件 | 将根目录外的原生依赖准确映射到模块；原始基类的外部库需安装在 ARTIQ 运行环境 |
| `resources` | `AOM_556_3DMOT`、`zotino0`、快门、共用 CPLD | 使用实际 `device_db` 器件名，逐项对照所有可达 kernel；DDS 和开关可能还有同一物理总线 |
| `build_kwargs` | `ew_v`、`offset_v`、`frequency_hz`、`attenuation_db` 等 | 键要与该类 `build(self, ...)` 的参数名一致；值在实例创建时固定。扫描量放到 stage 的调用处，例见 `GM_cooling(gm_frequency_hz)` |
| `initialize`、`after_shot`、`finalize` | 本例仅 MOT 选择 `initialize_devices` 与 `read_and_restore` | 分别为会话首次初始化、每个完成 shot 的结尾及会话最终收尾。异常可能中断执行，故障关断由真机入口负责 |
| `NativeProgram(modules=..., stages=...)` | `"gm" → GrayMolasses`，`"GM_cooling" → NativeStage("gm","cool", ...)` | 给实例取名，再将实验作者要调用的方法名映射到实例的 kernel。入参依照该 kernel 的签名 |
| `shared_resources` | `"urukul3_cpld": ("mot","gm","state")` 等 | 每一项列出全部参与者。只允许一个参与者做初始化，并让每一步自行建立需要的通道状态 |

本例 `gm_frequency_hz` 每次作为 `cool(self, GM_preparation_freq)` 的参数传入；
`gm_offset_v` 则在构造 GM 实例时设置为 `self.gray_EW_bias_offset`。
用同样的方法，可以把某个原本只在 Dashboard 中调整的量改为 DSL 扫描轴：先给薄适配 kernel
增加命名清晰的显式形参，再在 `exp.sweep(...)` 里将该 `SweepParameter` 传给对应的 stage。

In [ ]:
from taiyidsl import NativeModule, NativeProgram, NativeStage


def register_atom_program(master, adapter_root, values, base_source):
    source_overrides = {
        "lib": master / "lib/__init__.py",
        "lib._GreenMOTbase": master / base_file,
    }

    def component(cls, resources, settings, **hooks):
        # root 是生成文件的父目录；entry 用“模块:类”选中此组件的实现。
        # source_overrides 绑定该工作副本的 lib 与基类，工厂为每个源码文件记录哈希。
        # resources 枚举实际设备和共享总线；build_kwargs 将标量送入该类的 build()。
        # hooks 只选取需要的生命周期方法；没有指定的 run/prepare 均不调用。
        return NativeModule.from_package(
            root=adapter_root,
            entry=f"atom_stage_adapter:{cls}",
            source_overrides=source_overrides,
            resources=resources,
            build_kwargs=settings,
            **hooks,
        )

    # 基类 build() 静态绑定的物理器件；core 由根管理，scheduler 是虚拟设备。
    bound = {
        n.args[0].value
        for n in ast.walk(ast.parse(base_source))
        if isinstance(n, ast.Call)
        and isinstance(n.func, ast.Attribute)
        and n.func.attr == "setattr_device"
        and isinstance(n.args[0], ast.Constant)
    } - {"core", "scheduler"}
    board = "urukul3_cpld"  # 两个 556 DDS 通道还共享这一 CPLD/SPI
    # MOT 独占初始化权；ew_v 统一覆盖原基类默认偏置；tof_s 在读出前使用。
    mot = component(
        "MOT",
        sorted(bound | {board}),
        {"ew_v": values["ew_v"], "tof_s": values["tof_s"]},
        initialize="initialize_devices",
        after_shot="read_and_restore",
    )
    # GM 只声明自己使用的 556 MOT 输出、快门、Zotino 和共同板卡；不重复初始化。
    gm = component(
        "GrayMolasses",
        ("zotino0", "AOM_556_3DMOT", "ttl_556_3DMOT_Sh", board),
        {"ew_v": values["ew_v"], "offset_v": values["gm_offset_v"]},
    )
    # 态制备 build 接收抽运光频率/衰减以及偏置增量；源码片段使用这些对象属性。
    state = component(
        "OpticalPumping",
        ("zotino0", "AOM_556_Probe", board),
        {
            "ew_v": values["ew_v"],
            "offset_v": values["pump_offset_v"],
            "frequency_hz": values["pump_frequency_hz"],
            "attenuation_db": values["pump_att_db"],
        },
    )
    # modules 的键是组件实例名。stages 的键是实验作者真正调用的三个 DSL 方法。
    # 每个 NativeStage 依次指定：实例名、那个实例上的 @kernel 方法、时间边界类型。
    return NativeProgram(
        modules={"mot": mot, "gm": gm, "state": state},
        stages={
            "prepare_MOT": NativeStage("mot", "load", timing="prefix"),
            "GM_cooling": NativeStage("gm", "cool", timing="prefix"),
            "state_prep": NativeStage("state", "pump", timing="prefix"),
        },
        # 各设备列出全部使用者。三者共享 CPLD/SPI 和 Zotino；仅 mot 初始化。
        shared_resources={
            "zotino0": ("mot", "gm", "state"),
            board: ("mot", "gm", "state"),
            "AOM_556_3DMOT": ("mot", "gm"),
            "ttl_556_3DMOT_Sh": ("mot", "gm"),
            "AOM_556_Probe": ("mot", "state"),
        },
    )


production_native = None
if ATOM_MASTER is not None:
    production_native = register_atom_program(
        Path(ATOM_MASTER), adapter_root, lab_values, atom_sources[base_file]
    )
    print(production_native.to_dict()["lifecycle"])

In [ ]:
def production_shot(q, native, probe, awg_frequency, *, gm_frequency_hz, guard, shots):
    exp = Experiment("atom_control_578_spectroscopy", [q])
    # shots 决定重复数；awg_frequency 是电频率扫描轴；realtime 包含整个 shot。
    with exp.average(shots), exp.sweep(awg_frequency), exp.realtime():
        native.prepare_MOT()
        native.GM_cooling(gm_frequency_hz)  # 此处参数是经核对的 GM DDS 频率（Hz）
        native.state_prep()  # 光抽运时长已保存在原生 kernel 内
        # 外触发启动同一块 AWG；guard 是波形内的零输出时长（非原子端时延）。
        with exp.triggered_sequence(group="clock_awg"):
            delay(guard)
            play(q.clock, probe, frequency=awg_frequency)
        # 顶层 kernel 接下来调用 mot.read_and_restore()，发三次相机触发并收尾。
    return exp

生成的 `production_adapter/atom_stage_adapter.py` 可直接逐行对照上游文件。把它与
固定的 `lib/__init__.py`、`lib/_GreenMOTbase.py` 和完整导出目录一起交给 ARTIQ 环境；
worker 上两处模块根必须可导入且与导出的源路径、哈希匹配。迁移目录后应在最终路径重新导出。
此版本基类在导入时需要 `tifffile`。修改 GUI 入口、源码或接线后，应重新做离线编译。
基类会初始化 `AOM_578_Shift`，原实验另外驱动 `AOM_578_Clock`。外部 AWG 方案必须核对两者及目标射频链的实际开关/路由。
本组合仍沿用制备源中的光抽运与读出源中的 TOF。原钟谱的每点 OFF/ON 两次独立装载、
卫星 DDS 初始化及异常关光逻辑属于另一条运行链。本例生成单脉冲频谱；做配对实验时，
需显式安排 OFF/ON、每支的装载和相应帧编号，接入本实验台的失败关断策略。

### 用自己的适配层编译一份完整的混合导出

下一格有两种用法。只提供 `ATOM_MASTER` 时，用真实原生源码配合教学卡和
断开硬件的原生设备表，验证组合能否编译。再提供私有接线配置时，改为在
`reviewed_clock_setup` 上离线编译同一实验。两种用法都沿用教学频轴与探测幅度，
都需要在上机前填写实测的探测中心、单位换算和光学标定。

这里另存一份仅供编译器实例化驱动的 JSON：在仓库原有的合成 GreenMOT 表中，
按本版基类更新同一板卡位置的 759 路设备名，再给新快门分配一个教学 TTL。
这些通道号只属于断开硬件的编译测试，实验室接线由前面的私有快照和 descriptor 提供。

In [ ]:
if production_native is not None:
    old_fixture = native_root / "greenmot_native/device_db_offline.json"
    compile_devices = json.loads(old_fixture.read_text(encoding="utf-8"))
    compile_devices["AOM_759_Switch"] = compile_devices.pop("AOM_532_Switch")
    free_channel = 1 + max(
        row["arguments"]["channel"]
        for row in compile_devices.values()
        if isinstance(row, dict) and "channel" in row.get("arguments", {})
    )
    compile_devices["ttl_556_Probe_Sh"] = {
        "type": "local",
        "module": "artiq.coredevice.ttl",
        "class": "TTLOut",
        "arguments": {"channel": free_channel},
    }
    extra_db_path = OUT / "production_compile_devices.json"
    extra_db_path.write_text(json.dumps(compile_devices, indent=2), encoding="utf-8")

    if reviewed_clock_setup is None:
        # 只选 core 与演示触发线；原生快门和相机全部来自 GreenMOT 合成表。
        clock_db = {name: example_db[name] for name in ("core", "awg_trigger_b")}
        assert clock_db.keys().isdisjoint(compile_devices)
        production_setup = build_clock_setup(
            clock_db | compile_devices,
            fixture_cards,
            card_id="B",
            clock_port=0,
            trigger_name="awg_trigger_b",
        )
        print("原生输入：生产源码；接线：合成设备和教学 Spectrum 卡")
    else:
        production_setup = reviewed_clock_setup
        print("原生输入：生产源码；接线：已选实验室私有快照")
    # emulation=True 只编译；setup 与 q 的信号映射必须一致。
    production_lab = LabConfig(setup=production_setup, qubits=[q], emulation=True)
    production_exp = production_shot(
        q,
        production_native,
        probe,
        frequency_axis,
        gm_frequency_hz=lab_values["gm_frequency_hz"],
        guard=10 * us,
        shots=2,
    )
    production_compiled = production_lab.compile(production_exp)
    production_dir = OUT / "production_export"
    production_compiled.export(production_dir)
    plan = json.loads((production_dir / "artiq/manifest.json").read_text("utf-8"))
    native_plan = plan["native_composition"]
    assert native_plan["execution"] == "one_top_level_kernel_per_shot"
    assert len(native_plan["points"]) == len(drive_hz)
    print("ARTIQ 点 kernel 数", len(native_plan["points"]))
    print("完整混合导出", production_dir)

**用独立的 ARTIQ 9 解释器复核真实调用链。** 原生适配包导入上游
`lib._GreenMOTbase`，该文件在模块顶层导入 `tifffile`。令
`$atomMaster` 指向你刚才读取的同一个 `artiq-master`，确认 ARTIQ
解释器可以导入 `tifffile`。若实验室单独存放了这项依赖，也可把其
包的父目录作为另一个 `--module-root`。在 TaiyiDSL 的 PowerShell 中执行：

```powershell
$atomMaster = 'C:/lab/atom-control/artiq-master'
.\scripts\artiq-python.ps1 scripts/compile_native_artiq.py output/native_lab_tutorial/production_export/artiq `
    --output output/native_lab_tutorial/production_compile --target rv32g `
    --extra-device-db output/native_lab_tutorial/production_compile_devices.json `
    --module-root output/native_lab_tutorial/production_adapter --module-root $atomMaster
```
只把目标 CPU、驱动版本、输出代码和源指纹均对应本次实验的回执用于判断兼容性。
断开编译时使用模拟身份的本地驱动，现场还要核对卫星、板卡版本、实际延迟及采集路径。

## 怎样判断已经接通了哪一层

1. **实验编排**。先看导出清单的 `native_composition`，再读
   `artiq/experiment.py` 中的 `point_0` 及其调用的方法。确认每个 shot
   只装载一次，光抽运之后才有 AWG 触发，399 成像和收尾都只执行一次。
   查验 manifest 中的时基与源文件 SHA，保留匹配的 Python 包和波形文件。
2. **断开硬件的 ARTIQ 编译**。在已有项目 ARTIQ 9 工具链的 Windows
   checkout 运行下列命令。输入是上一节教学导出的 **ARTIQ 子目录**。
   `rv32g` 是本教程的离线目标；现场选择与 core 匹配的 target。

   ```powershell
   .\scripts\artiq-python.ps1 scripts/compile_native_artiq.py output/native_lab_tutorial/export/artiq `
       --output output/native_lab_tutorial/fixture_compile --target rv32g `
       --extra-device-db examples/artiq_spectrum/native_stages/device_db_offline.json --module-root examples/artiq_spectrum
   ```

   查看 `compile_report.json` 中的 `status`、`source_sha256`、每个 kernel 和 ELF。
   该命令使用 `core.host=None` 和合成设备表。得到 ELF 说明选定驱动与目标
   可完成代码生成；Spectrum 的实际触发延迟、RF 幅度和原子响应要分别测量。
3. **上机前的交接**。实验室入口应装入完整混合导出目录，通过匹配的
   `SpectrumRunner` 配置卡、上传并等待 ready，然后调用 ARTIQ 点 kernel。
   若通过 Dashboard，与 MOT 使用同一 master 的同一 pipeline。
   若使用 `lab.connect()`，它会启动独立进程直连设备，先让原实验释放
   core、卡和相机。保留实际的时钟会话、触发接线、现场标定与版本记录。
4. **读出钟谱**。用宿主 `result_adapter.before_shot` 使相机就绪，
   `after_shot` 按 point、shot、分支和帧序号关联三张图（原子＋光、光、暗场）。
   原脚本每个频点先做 OFF 参考，再重新装载做 ON。教学扫描只演示频率轴；
   加入参考时两分支保持相同装载、抽运、偏置和探测窗口，仅切换钟光。
   保存原始帧和缺帧状态，再用实验室标定的态选择读出估计激发分数。
   普通吸收图中的原子数下降还可能来自损失。直连 `HardwareSession.run()`
   没有 detector adapter 注入接口；需要帧时使用支持该读出适配器的宿主入口。

### 常见故障该查哪一处

| 看到的现象 | 检查位置和修正 |
|---|---|
| 找不到原生类、哈希改变或导入了另一版本 | 核对 `root` 是否指向顶层包的父目录、`entry` 的类名和 ARTIQ worker 的 import 路径。编辑上游文件后重新审查并导出 |
| 组件初始化冲突、共享别名或触发冲突 | 在实际设备库中将别名追到 DDS 通道、同板 CPLD/SPI 或 TTL。由一个组件初始化共享资源，给 Spectrum 选独占触发端口 |
| strict stage 中出现 `break_realtime`、RPC 或阻塞等待 | 沿调用链定位；把可变长准备放在 prefix，把 RPC 和数据传输放回宿主。若需要固定末边沿，另拆出可核对时长的 timed 尾段 |
| 触发早于准备结束或首个脉冲被截断 | 比对末边沿、同步组触发提前量和实测延迟；确认卡 ready，并在实验时序中留出所需间隔 |
| 生成了扫描点，示波器波形或原子图却完全相同 | 先比较各点最终 DAC 样本，再查 EXT0、模拟输出与 AOM 路由。最后对齐相机帧、OFF/ON 分支和频点索引 |

更换任何一个真实组件时，先单独复核它的输入、结束后的设备状态和时序边界，
再重新编译整条 shot。现场请优先比较原程序与组合程序在同一观测点的边沿；
尤其是 GM 末尾的偏置、光抽运关断、578 光脉冲及 399 相机首个下降沿。

[原生组件接口](../native_components.md) · [设备库与接线](../device_setup_descriptor.md) ·
[ARTIQ 工具链](../artiq_toolchain.md) · [相机读出接口](../readout_adapters.md)